# Step 1: Import Libraries
First, let's import the necessary libraries that will empower us to manipulate datasets, process text, and perform mathematical operations.

In [1]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import os

2025-04-30 23:03:16.734098: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Step2: Load the Dataset
We load the multi_news dataset, focusing on the 'test' split to efficiently manage our resources.


In [2]:
dataset = load_dataset("multi_news", split="test", trust_remote_code=True)
dataset

Dataset({
    features: ['document', 'summary'],
    num_rows: 5622
})

# Step 3: Data Preparation

In [3]:
df = dataset.to_pandas()

# Sample up to 10,000 or whatever is available
sample_size = min(len(df), 10000)
df = df.sample(sample_size, random_state=42)
df

,document,summary
4830,Tweet with a location \n \n You can add locati...,– Denis Finley has taken to Twitter to call Po...
1255,CNN host Piers Morgan just called to discuss h...,– CNN's Piers Morgan thinks gun-rights propone...
80,White House communications director Anthony Sc...,– New White House communications director Anth...
3044,CLOSE Scientists say they've found archaeologi...,– Scientists say they have the first physical ...
4486,Click image above to view graphic \n \n Althou...,– Scientists are calling it a breakthrough and...
...,...,...
3772,Danny Iudici/New York Daily News Union members...,– It's a rough day to be a schoolkid in the Bi...
5191,"As we reported earlier, LulzSec says that it h...",– Just as it looked like LulzSec was gearing u...
5226,Remember that movie with Nic Cage where he had...,– Different types of cars rotate in and out of...
5390,"An experimental performance unveils the ""wizar...",– Lady Gaga showed up on the Grammys red carpe...


# Step 4: Load the Model
Next, we load a pre-trained Sentence Transformer model. This model will help us convert textual data into dense vectors (embeddings) that capture the essence of our text.

In [4]:
# Load the sentence transformer model (lightweight, fast)
model = SentenceTransformer("all-MiniLM-L6-v2")
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# Step 5: Generate Embeddings
Here, we encode the article summaries into embeddings, transforming the textual information into a numerical format that's easier to analyze.

In [5]:
# Encode summaries (convert_to_tensor=False = NumPy for memory safety)
passage_embeddings = model.encode(
    df["summary"].tolist(),
    show_progress_bar=True,
    convert_to_tensor=False
)

# Convert to NumPy array if not already
passage_embeddings = np.array(passage_embeddings)
passage_embeddings[0].shape


Batches:   0%|          | 0/176 [00:00<?, ?it/s]

(384,)

# Step 6: Find Relevant Articles
To find articles that match our query, we compute the cosine similarity between the query embedding and all the article embeddings, retriving the top 3 most relevant articles.

In [6]:
# Function to find top relevant news
def find_relevant_news(query, top_k=3):
    query_embedding = model.encode(query, convert_to_tensor=False)
    similarities = cosine_similarity([query_embedding], passage_embeddings)[0]
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    return [df.iloc[i]["summary"][:200] + "..." for i in top_indices]

In [7]:
# Example1 usage
results = find_relevant_news("Natural disasters")
for i, summary in enumerate(results, 1):
    print(f"\nTop {i}:\n{summary}")


Top 1:
– Harvey is getting its proper attention in the US, but another devastating flood is unfolding in a different part of the world. Monsoon rains have triggered flooding and mudslides that have left more...

Top 2:
– The tsunami that killed hundreds, possibly thousands of people after an earthquake in Indonesia on Friday was much bigger and more devastating than would normally be expected after that kind of quak...

Top 3:
– A rare outbreak of winter tornadoes has killed at least seven people in Missouri and Arkansas and left a trail of destruction across the South and Midwest. Three people were killed by a tornado in a...


In [8]:
# Example2
results = find_relevant_news("Politics, diplomacy and nationalism")
for i, summary in enumerate(results, 1):
    print(f"\nTop {i}:\n{summary}")


Top 1:
– President Obama addressed the United Nations as president for the final time on Tuesday with a speech that urged world leaders make a "course correction" and be more open to refugees and less author...

Top 2:
– President Obama is in Cambodia today, meeting with East Asian leaders amid a contentious territorial dispute over the South China Sea. Speaking publicly, neither Obama nor Chinese Premier Wen Jiabao...

Top 3:
– He's known for his humility, his down-to-earth nature, his personal phone calls to the flock, and even his selfies. But is Pope Francis actually a "political genius"? Yes, writes Candida Moss at Pol...


# Conclusion and Next Steps
This notebook illustrates the power of NLP in extracting relevant information from large text datasets using sentence embeddings and cosine similarity. 

In [9]:
def clear_screen():
    os.system("clear")

In [10]:
def interactive_search():
    print("Welcome to the Semantic News Search!\n")
    while True:
        print("Type in a topic you'd like to find articles about, and I'll do the searching! (Type 'exit' to quit)\n ", end="")
        query = input().strip()
        
        if query.lower() == "exit":
            print("\nThanks for using the Semantic News Search! Have a great day!")
            break
        
        print("\n\tHere are 3 articles I found based on your query: \n")
        
        passages = find_relevant_news(query)
        for passage in passages:
            print("\n\t"+passage)
        
        input("\nPress Enter to continue searching...")
        clear_screen()

In [12]:
# Start the interactive search
interactive_search()

Welcome to the Semantic News Search!

Type in a topic you'd like to find articles about, and I'll do the searching! (Type 'exit' to quit)
 Canada US relationship ?

	Here are 3 articles I found based on your query: 


	– Looking for true love? Thinking of fleeing to Canada if Donald Trump wins the general election in November? Now you can take care of both in one fell swoop thanks to a site that promises to "make da...

	– The traditional end-of-summit group photo at this year's G7 gathering will not include President Trump. The White House says Trump will leave the Quebec summit on Saturday morning and travel directl...

	– Reactions to the biggest political upset in recent memory are pouring in from around the globe, and if any world leaders are shaking in their boots, they aren't showing it. A look around: Mexico: "I...

Press Enter to continue searching...exit
Type in a topic you'd like to find articles about, and I'll do the searching! (Type 'exit' to quit)
 exit

Thanks for using